## f0 Search Utility (`search_f0`)

`search_f0` detects the f0 that best explains measured frequencies as integer partials.

### Parameter Reference

- `freqs`: list/array of measured frequencies in Hz.
- `f0_min`: lower bound of f0 search range in Hz.
- `f0_max`: upper bound of f0 search range in Hz.
- `step`: fine search step in Hz.
- `peaks_df`: optional dataframe input (e.g. from `analyze_signal`) as an alternative to `freqs`.
- `freqs_col`: frequency column name in `peaks_df`.
- `weights`: optional manual weight array (same length as frequencies).
- `use_amp_weights`: use `peaks_df[amp_col]` as weights.
- `amp_col`: amplitude column name for automatic weighting.
- `top_k`: number of best candidates returned.
- `trim_outlier_cents`: drop residuals above this cents threshold.
- `robust_trim_frac`: drop this fraction of worst residuals per candidate.
- `coarse_step`: optional coarse global search step before refinement.
- `refine_half_width_hz`: half-width of fine refinement band around best coarse f0.
- `include_plot`: include a Plotly RMS-cents search curve in the result.
- `show_plot`: render the plot immediately.
- `annotate_plot`: add search-parameter annotations to the plot.
- `plot_title`: Plotly title.
- `plot_width`: plot width in pixels.
- `plot_height`: plot height in pixels.
- `plotly_layout`: optional layout overrides passed to Plotly.

### Parameter Annotations (quick guidance)

- `step`: smaller values are more precise, but increase compute time.
- `top_k`: useful for ambiguous spectra where several nearby f0 values are plausible.
- `weights` / `use_amp_weights`: emphasize reliable/strong partials.
- `trim_outlier_cents` + `robust_trim_frac`: stabilize detection when one or more partials are noisy or spurious.
- `coarse_step`: speeds up wide-range search; use with `refine_half_width_hz` for accurate final estimates.

### Parameter Reference: `summarize_result(...)`

- `label`: title text printed above the summary block.
- `result`: dictionary returned by `search_f0(...)`.
- `top_n`: how many top candidates to print.
- `precision`: number of decimal digits used in formatted numeric output.
- `suppress`: if `True`, suppresses very small values in NumPy-style printing.
- `floatmode`: NumPy float formatting mode (e.g. `"fixed"`, `"maxprec"`).
- `show_plot`: if `True`, render the stored Plotly figure from `result["fig"]`.
- `plot_width`: optional width override applied to the stored figure before display.
- `plot_height`: optional height override applied to the stored figure before display.
- `plotly_layout`: optional Plotly layout overrides applied before display.


In [ ]:
import numpy as np
from audiospylt import search_f0, summarize_result

#np.set_printoptions(precision=3, suppress=True, floatmode="fixed")

freqs = [500.4, 565.0, 706.2, 918.2, 988.8]
plot_width = 1200

result = search_f0(
    freqs=freqs,
    f0_min=35.0,
    f0_max=150.0,
    step=0.001,
    top_k=5,
    include_plot=True,
    show_plot=False,
    annotate_plot=True,
    plot_width=plot_width,
)

summarize_result(
    "Baseline list input",
    result,
    top_n=5,
    precision=3,
    suppress=True,
    floatmode="fixed",
    show_plot=True,
    plot_width=plot_width,
)


### Use Case 1: DataFrame integration (`peaks_df`) with amplitude weighting

Here we pass a DataFrame directly and weight candidates using `Amplitude`.


In [ ]:
import pandas as pd

peaks_df = pd.DataFrame(
    {
        "Frequency (Hz)": [500.4, 565.0, 706.2, 918.2, 988.8],
        "Amplitude": [0.82, 0.55, 0.28, 0.20, 0.35],
    }
)

df_result = search_f0(
    peaks_df=peaks_df,
    freqs_col="Frequency (Hz)",
    use_amp_weights=True,
    amp_col="Amplitude",
    f0_min=35.0,
    f0_max=150.0,
    step=0.001,
    top_k=3,
    include_plot=True,
    plot_width=plot_width,
)

summarize_result("DataFrame + amplitude weights", df_result, precision=3)


### Use Case 2: Manual weights and robust outlier handling

This example includes one outlier frequency and compares plain search vs robust settings.


In [ ]:
freqs_with_outlier = [500.4, 565.0, 706.2, 918.2, 988.8, 1234.0]
manual_weights = [1.0, 1.0, 1.0, 1.0, 1.0, 0.2]  # de-emphasize outlier

plain = search_f0(
    freqs=freqs_with_outlier,
    f0_min=35.0,
    f0_max=150.0,
    step=0.001,
    top_k=5,
    include_plot=True,
    plot_width=plot_width,
)

robust = search_f0(
    freqs=freqs_with_outlier,
    f0_min=35.0,
    f0_max=150.0,
    step=0.001,
    top_k=5,
    weights=manual_weights,
    trim_outlier_cents=35.0,
    robust_trim_frac=0.20,
    include_plot=True,
    plot_width=plot_width,
)

summarize_result("Plain search (with outlier)", plain)
summarize_result("Robust search (weights + trimming)", robust)


### Use Case 3: Coarse-to-fine search

Use `coarse_step` for speed over large ranges, then refine around the best coarse candidate.


In [ ]:
coarse_fine = search_f0(
    freqs=freqs,
    f0_min=20.0,
    f0_max=220.0,
    step=0.001,
    coarse_step=0.05,
    refine_half_width_hz=0.5,
    top_k=5,
    include_plot=True,
    annotate_plot=True,
    plot_width=plot_width,
)

summarize_result("Coarse-to-fine search", coarse_fine, top_n=5)
print(f"candidate grid size: {len(coarse_fine['f0_candidates_hz'])}")
